# Rental Product Recommendation: preprocessing pipeline

Цель: собрать минимальный офлайн-пайплайн подготовки данных для задач
сессионных рекомендаций (Recall@6). Мы готовим:

- связи `watch_id -> visit_id`
- отображение `slug -> product_id` нового сайта
- последовательности `product_id` в рамках каждой сессии

Файлы должны лежать в `datasets/`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd

DATA_DIR = Path('datasets')
OUT_DIR = Path('processed')
OUT_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [2]:
# Load datasets
hits = pd.read_csv(DATA_DIR / 'metrika_hits.csv', low_memory=False)
visits = pd.read_csv(DATA_DIR / 'metrika_visits.csv', low_memory=False)

hits_test = pd.read_csv(DATA_DIR / 'metrika_hits_test.csv', low_memory=False)
visits_test = pd.read_csv(DATA_DIR / 'metrika_visits_test.csv', low_memory=False)

new_products = pd.read_csv(DATA_DIR / 'new_site_products.csv', low_memory=False)
mapping_old_new = pd.read_csv(DATA_DIR / 'old_site_new_site_products.csv', low_memory=False)

print('hits', hits.shape, 'visits', visits.shape)
print('hits_test', hits_test.shape, 'visits_test', visits_test.shape)
print('new_products', new_products.shape, 'mapping', mapping_old_new.shape)


hits (1721596, 40) visits (323241, 26)
hits_test (23260, 40) visits_test (3891, 26)
new_products (665, 96) mapping (464, 2)


In [3]:
# Build watch_id -> visit_id mapping (visits.watch_ids is a JSON array of strings)
def parse_watch_ids(value: str) -> list[str]:
    if pd.isna(value) or value == '[]':
        return []
    try:
        return json.loads(value)
    except Exception:
        return []

def build_watch_to_visit(df_visits: pd.DataFrame) -> pd.DataFrame:
    tmp = df_visits[['visit_id', 'watch_ids']].copy()
    tmp['watch_ids'] = tmp['watch_ids'].map(parse_watch_ids)
    tmp = tmp.explode('watch_ids')
    tmp = tmp.rename(columns={'watch_ids': 'watch_id'})
    tmp = tmp.dropna(subset=['watch_id'])
    return tmp

watch_to_visit = build_watch_to_visit(visits)
watch_to_visit_test = build_watch_to_visit(visits_test)

print('watch_to_visit', watch_to_visit.shape)
print('watch_to_visit_test', watch_to_visit_test.shape)


watch_to_visit (2757131, 2)
watch_to_visit_test (41920, 2)


In [4]:
# Map slug -> product_id for the new site
def normalize_slug(value: str) -> str | None:
    if pd.isna(value):
        return None
    return str(value).strip().lower()

new_products['slug_norm'] = new_products['slug'].map(normalize_slug)
slug_to_product = new_products.dropna(subset=['slug_norm'])[['slug_norm', 'id']]
slug_to_product = slug_to_product.drop_duplicates('slug_norm')
slug_to_product = slug_to_product.rename(columns={'id': 'product_id'})

def extract_slug_from_url(value: str) -> str | None:
    if pd.isna(value):
        return None
    try:
        path = urlparse(str(value)).path
        if not path:
            return None
        parts = [p for p in path.split('/') if p]
        return parts[-1].lower() if parts else None
    except Exception:
        return None

def add_product_id(df_hits: pd.DataFrame) -> pd.DataFrame:
    tmp = df_hits.copy()
    tmp['date_time'] = pd.to_datetime(tmp['date_time'], errors='coerce')
    tmp['slug_norm'] = tmp['slug'].map(normalize_slug)
    # If slug missing, try extract from URL
    missing_slug = tmp['slug_norm'].isna()
    tmp.loc[missing_slug, 'slug_norm'] = tmp.loc[missing_slug, 'url'].map(extract_slug_from_url)
    tmp = tmp.merge(slug_to_product, on='slug_norm', how='left')
    return tmp

hits = add_product_id(hits)
hits_test = add_product_id(hits_test)

print('hits with product_id', hits['product_id'].notna().mean())
print('hits_test with product_id', hits_test['product_id'].notna().mean())


hits with product_id 0.051137432940132295
hits_test with product_id 0.2766122098022356


In [5]:
# Join hits -> visit_id and build session sequences
def build_sequences(df_hits: pd.DataFrame, watch_map: pd.DataFrame) -> pd.DataFrame:
    tmp = df_hits.merge(watch_map, on='watch_id', how='left')
    tmp = tmp.dropna(subset=['visit_id'])
    # Keep only product page views that map to new_site product_id
    tmp = tmp[tmp['page_type'] == 'PRODUCT']
    tmp = tmp.dropna(subset=['product_id'])
    tmp = tmp.sort_values(['visit_id', 'date_time'])
    sequences = tmp.groupby('visit_id')['product_id'].apply(list).reset_index()
    sequences['session_len'] = sequences['product_id'].map(len)
    return sequences

train_sequences = build_sequences(hits, watch_to_visit)
test_sequences = build_sequences(hits_test, watch_to_visit_test)

print('train_sequences', train_sequences.shape)
print('test_sequences', test_sequences.shape)
train_sequences.head()


ValueError: You are trying to merge on uint64 and object columns for key 'watch_id'. If you wish to proceed you should use pd.concat

In [ ]:
# Save artifacts for later stages (candidate generation / ranking)
train_sequences.to_parquet(OUT_DIR / 'train_sequences.parquet', index=False)
test_sequences.to_parquet(OUT_DIR / 'test_sequences.parquet', index=False)

watch_to_visit.to_parquet(OUT_DIR / 'watch_to_visit.parquet', index=False)
watch_to_visit_test.to_parquet(OUT_DIR / 'watch_to_visit_test.parquet', index=False)

print('Saved to', OUT_DIR)
